# 풀스택 GPT: #4.0~4.6

## Tasks:

- [x] 영화 이름을 가지고 감독, 주요 출연진, 예산, 흥행 수익, 영화의 장르, 간단한 시놉시스 등 영화에 대한 정보로 답장하는 체인을 만드세요.
- [x] LLM은 항상 동일한 형식을 사용하여 응답해야 하며, 이를 위해서는 원하는 출력의 예시를 LLM에 제공해야 합니다.
    > - [x] LLM이 답변 형식을 효과적으로 학습하려면 모든 예시는 동일한 형식을 유지해야 합니다.
- [x] 예제를 제공하려면 [FewShotPromptTemplate](https://python.langchain.com/v0.1/docs/modules/model_io/prompts/few_shot_examples/) 또는 [FewShotChatMessagePromptTemplate](https://python.langchain.com/v0.1/docs/modules/model_io/prompts/few_shot_examples_chat/)을 사용하세요.

In [3]:
examples = [
    {
        "title": "The Sentimental Policeman",
        "synopsis": "In this affectionate, leisurely paced comedy, an Odessa policeman is out walking his beat when he discovers an adorable infant abandoned in a cabbage patch. He does his duty and takes the baby to an orphanage, but later he and his wife, who have an unusually affectionate and cozy relationship, decide to try and adopt the little one. What they must go through to accomplish that goal is anything but straightforward.",
        "year": 1992,
        "rate": 3.8,
        "genre": ["Comedy"],
        "director": "Kira Muratova",
        "language": "Russian",
    },
    {
        "title": "Nem Sansão Nem Dalila",
        "synopsis": "Barber's jeep crash against crazy scientist's house, where the latter was building a time-machine. The crash triggers the machine, taking them to Gaza kingdom, circa 1153 B.C., where they get involved in many funny situations. Spoof of Cecil B. DeMille's Samson and Delilah",
        "year": 1955,
        "rate": 6.3,
        "genre": ["Comedy", "Science Fiction",],
        "director": "Carlos Manga",
        "language": "Portuguese",
    },
    {
        "title": "Harmony",
        "synopsis": "Jeong-hye gives a birth to a baby boy in the prison. Her and other inmates create a women's choir to compete in the national choir contest, to meet and greet their families and loving ones.",
        "year": 2010,
        "rate": 7.677,
        "genre": ["Drama", "Music",],
        "director": "Kang Dae-gyu",
        "language": "Korean",
    },
]

In [6]:
from langchain.chat_models.openai import ChatOpenAI
from langchain.prompts import PromptTemplate, FewShotPromptTemplate

llm = ChatOpenAI(temperature=0.1)

example_template = """
    System: You are the movie geek. If the movie title is given, you answer all you know about in a format.
    Human: What do you know about the movie, {title}?
    AI: 
        "director":{director},
        "genre":{genre},
        "synopsis":{synopsis},
        "year":{year},
        "rate":{rate},
        "language":{language},
"""

example_prompt = PromptTemplate.from_template(example_template)

prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Human: What do you know about the movie, {title}?",
    input_variables=["title"],
)

chain = prompt | llm
chain.invoke({
    "title":"PARASITE"
})

AIMessage(content='AI: \n        "director":Bong Joon-ho,\n        "genre":[\'Drama\', \'Thriller\'],\n        "synopsis":A poor family, the Kims, con their way into becoming the servants of a rich family, the Parks. But their easy life gets complicated when their deception is threatened to be exposed.,\n        "year":2019,\n        "rate":8.6,\n        "language":Korean,')

In [5]:
from langchain.chat_models.openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

llm = ChatOpenAI(temperature=0.1)

example_template = """
    "director":{director},
    "genre":{genre},
    "synopsis":{synopsis},
    "year":{year},
    "rate":{rate},
    "language":{language},
"""

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "What do you know about the movie, {title}?"),
    ("ai", example_template),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are the movie geek. If the movie title is given, you answer all you know about in a format."),
    few_shot_prompt,
    ("human", "What do you know about the movie, {title}?"),
])

chain = final_prompt | llm
chain.invoke({
    "title":"Moonrise Kingdom"
})

AIMessage(content='\n    "director":Wes Anderson,\n    "genre":[\'Adventure\', \'Comedy\', \'Drama\'],\n    "synopsis":A pair of young lovers flee their New England town, which causes a local search party to fan out to find them.,\n    "year":2012,\n    "rate":7.8,\n    "language":English,\n    "cast":["Jared Gilman", "Kara Hayward", "Bruce Willis", "Edward Norton", "Bill Murray", "Frances McDormand", "Tilda Swinton", "Jason Schwartzman", "Bob Balaban"]')